In [2]:
import os
import subprocess
from PyLTSpice import RawRead

# 1. SETTINGS
LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
base_dir = r"C:\Users\Mohamed\Desktop\ML\Test_someting"
netlist_path = os.path.join(base_dir, "Res_divider.txt")
raw_path = os.path.join(base_dir, "Res_divider.raw")

# 2. RUN SIMULATION
# -Run starts it, -b makes it a background process
print("Running LTspice 24 Engine...")
subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], check=True)

# 3. READ THE BINARY OUTPUT (.raw file)
if os.path.exists(raw_path):
    LTR = RawRead(raw_path)
    
    # Get the data for your nodes
    # LTspice stores these in the 'trace' list
    vin_data = LTR.get_trace("V(vin)").get_wave()
    vout_data = LTR.get_trace("V(vout)").get_wave()
    ir1_data = LTR.get_trace("I(R1)").get_wave()

    # Since it's a .op simulation, the first value [0] is your result
    v_in = vin_data[0]
    v_out = vout_data[0]
    i_r1 = ir1_data[0]

    print("\n--- DATA ACCESSED DIRECTLY FROM BINARY ---")
    print(f"Input Voltage  (vin)  : {v_in:.4f} V")
    print(f"Output Voltage (vout) : {v_out:.4f} V")
    print(f"Current through R1    : {i_r1:.6f} A")
    print(f"Calculated Gain       : {v_out/v_in:.2f}")
else:
    print(f"Error: Could not find {raw_path}. Check if LTspice actually ran.")

Running LTspice 24 Engine...

--- DATA ACCESSED DIRECTLY FROM BINARY ---
Input Voltage  (vin)  : 10.0000 V
Output Voltage (vout) : 5.0000 V
Current through R1    : 0.000500 A
Calculated Gain       : 0.50


In [5]:
import os
import subprocess
import numpy as np
from PyLTSpice import RawRead

# 1. SETTINGS
LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
base_dir = r"C:\Users\Mohamed\Desktop\ML\Test_someting"
netlist_path = os.path.join(base_dir, "cs_amp.sp") # Rename if necessary
raw_path = os.path.join(base_dir, "cs_amp.raw")

# 2. RUN SIMULATION
print("Simulating MOSFET AC Response...")
subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], check=True)

# 3. READ THE BINARY OUTPUT
if os.path.exists(raw_path):
    LTR = RawRead(raw_path)
    
    # Get the traces
    # Note: AC results are complex numbers (magnitude and phase)
    freqs = LTR.get_trace("frequency").get_wave()
    vout_complex = LTR.get_trace("V(out)").get_wave()
    
    # Calculate Magnitude
    magnitudes = np.abs(vout_complex)
    
    # --- CALCULATE GAIN ---
    # Max magnitude in the frequency range
    gain_val = np.max(magnitudes)
    
    # --- CALCULATE BANDWIDTH (3dB point) ---
    target_mag = gain_val / np.sqrt(2)
    # Find the frequency where magnitude drops below target
    # We look for the first index where magnitude < target after the peak
    try:
        idx_bw = np.where(magnitudes <= target_mag)[0][0]
        bw_val = np.abs(freqs[idx_bw]) # freqs are sometimes stored as complex
    except IndexError:
        bw_val = float('inf') # If it never drops, BW is beyond sweep range

    # 4. STORE IN FINAL VARIABLES
    gain_python = gain_val
    bw_python = bw_val

    print("\n--- MOSFET Analysis Results ---")
    print(f"Gain (max): {gain_python:.4f}")
    print(f"Bandwidth:  {bw_python/1e6:.2f} MHz")
    
else:
    print(f"Error: {raw_path} not found.")

Simulating MOSFET AC Response...

--- MOSFET Analysis Results ---
Gain (max): 22.4390
Bandwidth:  5.89 MHz


In [8]:
import os
import subprocess
import numpy as np
from PyLTSpice import RawRead

def run_simulation(L, W, VG, Rd):
    # 1. SETTINGS
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Test_someting"
    netlist_path = os.path.join(base_dir, "cs_amp.sp")
    raw_path = os.path.join(base_dir, "cs_amp.raw")

    # 2. UPDATE PARAMETERS IN THE NETLIST
    # Read the existing file
    with open(netlist_path, 'r') as f:
        lines = f.readlines()

    # Create new content with updated parameters
    new_lines = []
    for line in lines:
        upper_line = line.upper().strip()
        if upper_line.startswith(".PARAM RD="):
            new_lines.append(f".param Rd={Rd}\n")
        elif upper_line.startswith(".PARAM WN="):
            new_lines.append(f".param Wn={W}\n")
        elif upper_line.startswith(".PARAM VG="):
            new_lines.append(f".param VG={VG}\n")
        elif upper_line.startswith(".PARAM LN="):
            new_lines.append(f".param Ln={L}\n")
        else:
            new_lines.append(line)

    # Write the modified netlist back to disk
    with open(netlist_path, 'w') as f:
        f.writelines(new_lines)

    # 3. RUN SIMULATION
    subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], check=True)

    # 4. DATA EXTRACTION
    if os.path.exists(raw_path):
        LTR = RawRead(raw_path)
        freqs = LTR.get_trace("frequency").get_wave()
        vout_complex = LTR.get_trace("V(out)").get_wave()
        magnitudes = np.abs(vout_complex)

        # Gain calculation
        gain = np.max(magnitudes)
        
        # Bandwidth calculation
        target_mag = gain / np.sqrt(2)
        try:
            idx_bw = np.where(magnitudes <= target_mag)[0][0]
            bw = np.abs(freqs[idx_bw])
        except IndexError:
            bw = 0 # Out of range

        return gain, bw
    return None, None

# --- MAIN EXECUTION ---
# Input your new parameters here
new_L  = 5e-06
new_W  = 0.0002
new_VG = 1.1
new_Rd = 25000.0

gain, bw = run_simulation(new_L, new_W, new_VG, new_Rd)

print("\n--- Simulation Results with New Parameters ---")
print(f"L: {new_L}, W: {new_W}, VG: {new_VG}, Rd: {new_Rd}")
print(f"Gain: {gain:.4f}")
print(f"BW:   {bw/1e6:.2f} MHz")


--- Simulation Results with New Parameters ---
L: 5e-06, W: 0.0002, VG: 1.1, Rd: 25000.0
Gain: 4.7607
BW:   15.85 MHz
